## 1. Setup & Imports

In [1]:
import numpy as np
import pandas as pd
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score
from scipy.optimize import minimize
import optuna
from optuna.samplers import TPESampler
import warnings, os, gc, time, torch

warnings.filterwarnings('ignore')
optuna.logging.set_verbosity(optuna.logging.WARNING)

import lightgbm as lgb
import catboost as cb
import xgboost as xgb

SEED = 42
N_FOLDS = 5
np.random.seed(SEED)

# ----- GPU Detection -----
HAS_GPU = torch.cuda.is_available()
print(f"GPU Available: {HAS_GPU}")

if HAS_GPU:
    print(f"Hardware: {torch.cuda.get_device_name(0)}")
    LGBM_DEVICE = "gpu"
    XGB_TREE_METHOD = "gpu_hist" # or "device='cuda'" for newer versions
    CAT_TASK_TYPE = "GPU"
else:
    print("No GPU detected, using CPU fallback.")
    LGBM_DEVICE = "cpu"
    XGB_TREE_METHOD = "hist"
    CAT_TASK_TYPE = "CPU"

print("All libraries loaded")

GPU Available: True
Hardware: Tesla T4
All libraries loaded


## 2. Load Data

In [3]:
if os.path.exists('/kaggle/input'):
    BASE_PATH = "/kaggle/input/competitions/playground-series-s6e3/"
    print(f"Kaggle environment: {BASE_PATH}")
else:
    BASE_PATH = 'ChurnMarch/'
    print(f"Local environment: {BASE_PATH}")

train_df = pd.read_csv(os.path.join(BASE_PATH, 'train.csv'))
test_df = pd.read_csv(os.path.join(BASE_PATH, 'test.csv'))
sample_sub = pd.read_csv(os.path.join(BASE_PATH, 'sample_submission.csv'))

print(f"Train: {train_df.shape}, Test: {test_df.shape}")
print(f"Target distribution:\n{train_df['Churn'].value_counts(normalize=True)}")

Kaggle environment: /kaggle/input/competitions/playground-series-s6e3/
Train: (594194, 21), Test: (254655, 20)
Target distribution:
Churn
No     0.774792
Yes    0.225208
Name: proportion, dtype: float64


In [4]:
train_df.head()

,id,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,0,Male,0,Yes,Yes,29,Yes,No,DSL,Yes,...,Yes,Yes,No,No,One year,Yes,Mailed check,60.10,1653.85,No
1,1,Male,0,Yes,Yes,58,Yes,No,DSL,Yes,...,No,Yes,Yes,No,Two year,No,Credit card (automatic),69.50,3778.20,No
2,2,Male,0,Yes,No,58,Yes,Yes,Fiber optic,No,...,No,No,Yes,Yes,Month-to-month,Yes,Electronic check,100.40,5841.35,No
3,3,Female,0,No,No,1,Yes,No,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,69.70,70.70,Yes
4,4,Female,0,No,No,1,Yes,No,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,70.45,70.45,Yes


In [5]:
train_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 594194 entries, 0 to 594193
Data columns (total 21 columns):
 #   Column            Non-Null Count   Dtype  
---  ------            --------------   -----  
 0   id                594194 non-null  int64  
 1   gender            594194 non-null  object 
 2   SeniorCitizen     594194 non-null  int64  
 3   Partner           594194 non-null  object 
 4   Dependents        594194 non-null  object 
 5   tenure            594194 non-null  int64  
 6   PhoneService      594194 non-null  object 
 7   MultipleLines     594194 non-null  object 
 8   InternetService   594194 non-null  object 
 9   OnlineSecurity    594194 non-null  object 
 10  OnlineBackup      594194 non-null  object 
 11  DeviceProtection  594194 non-null  object 
 12  TechSupport       594194 non-null  object 
 13  StreamingTV       594194 non-null  object 
 14  StreamingMovies   594194 non-null  object 
 15  Contract          594194 non-null  object 
 16  PaperlessBilling  59

## 3. Feature Engineering

In [6]:
def feature_engineering(train, test):

    train = train.copy()
    test = test.copy()

    for df in [train, test]:
        df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')
        df['TotalCharges'] = df['TotalCharges'].fillna(df['MonthlyCharges'])

    train['Churn_encoded'] = train['Churn'].map({'Yes': 1, 'No': 0})

    target_encode_cols = [
        'gender', 'Partner', 'Dependents', 'PhoneService', 'MultipleLines',
        'InternetService', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection',
        'TechSupport', 'StreamingTV', 'StreamingMovies', 'Contract',
        'PaperlessBilling', 'PaymentMethod'
    ]

    global_mean = train['Churn_encoded'].mean()
    skf_te = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
    smooth_factor = 20

    for col in target_encode_cols:
        train[f'{col}_te'] = global_mean
        test_mapping = train.groupby(col)['Churn_encoded'].mean().to_dict()

        for tr_idx, val_idx in skf_te.split(train, train['Churn_encoded']):
            fold_mapping = train.iloc[tr_idx].groupby(col)['Churn_encoded'].mean()
            fold_counts = train.iloc[tr_idx].groupby(col)['Churn_encoded'].count()
            smoothed = (fold_mapping * fold_counts + global_mean * smooth_factor) / (fold_counts + smooth_factor)
            train.loc[train.index[val_idx], f'{col}_te'] = (
                train.iloc[val_idx][col].map(smoothed).fillna(global_mean)
            )

        test[f'{col}_te'] = test[col].map(test_mapping).fillna(global_mean)

    freq_encode_cols = ['InternetService', 'Contract', 'PaymentMethod']
    for col in freq_encode_cols:
        freq_map = train[col].value_counts(normalize=True).to_dict()
        train[f'{col}_freq'] = train[col].map(freq_map)
        test[f'{col}_freq'] = test[col].map(freq_map).fillna(0)

    # ===== NUMERIC & DERIVED FEATURES =====
    for df in [train, test]:
        t = df['tenure'].astype(float)
        mc = df['MonthlyCharges']
        tc = df['TotalCharges']

        # Tenure features
        df['tenure_f'] = t
        df['tenure_log'] = np.log1p(t)
        df['tenure_sq'] = t ** 2
        df['tenure_sqrt'] = np.sqrt(t)
        df['is_new'] = (t <= 3).astype(int)
        df['is_loyal'] = (t >= 48).astype(int)
        df['is_mid_tenure'] = ((t > 12) & (t <= 36)).astype(int)
        df['tenure_bin5'] = pd.cut(t, bins=[-1, 6, 12, 24, 48, 72, 200], labels=False)

        # Charges features
        df['avg_monthly'] = tc / (t + 1)
        df['charge_ratio'] = mc / (df['avg_monthly'] + 1e-5)
        df['charge_increase'] = mc - df['avg_monthly']
        df['charge_increase_pct'] = df['charge_increase'] / (df['avg_monthly'] + 1e-5)
        df['remaining_value'] = mc * (72 - t).clip(lower=0)
        df['expected_total'] = mc * t
        df['total_diff'] = tc - df['expected_total']
        df['total_diff_pct'] = df['total_diff'] / (df['expected_total'] + 1e-5)
        df['mc_bin5'] = pd.cut(mc, bins=[0, 30, 50, 70, 90, 200], labels=False)
        df['is_high_charge'] = (mc > 70).astype(int)
        df['is_low_charge'] = (mc < 30).astype(int)
        df['mc_log'] = np.log1p(mc)
        df['tc_log'] = np.log1p(tc)
        df['mc_tc_ratio'] = mc / (tc + 1e-5)

        # Service count features
        internet_svcs = ['OnlineSecurity', 'OnlineBackup', 'DeviceProtection',
                         'TechSupport', 'StreamingTV', 'StreamingMovies']
        df['n_internet_svcs'] = sum((df[c] == 'Yes').astype(int) for c in internet_svcs)
        df['has_phone'] = (df['PhoneService'] == 'Yes').astype(int)
        df['has_internet'] = (df['InternetService'] != 'No').astype(int)
        df['has_fiber'] = (df['InternetService'] == 'Fiber optic').astype(int)
        df['has_dsl'] = (df['InternetService'] == 'DSL').astype(int)
        df['has_security'] = (df['OnlineSecurity'] == 'Yes').astype(int)
        df['has_backup'] = (df['OnlineBackup'] == 'Yes').astype(int)
        df['has_protection'] = (df['DeviceProtection'] == 'Yes').astype(int)
        df['has_support'] = (df['TechSupport'] == 'Yes').astype(int)
        df['n_protect'] = df['has_security'] + df['has_backup'] + df['has_protection'] + df['has_support']
        df['n_stream'] = sum((df[c] == 'Yes').astype(int) for c in ['StreamingTV', 'StreamingMovies'])
        df['total_svcs'] = df['has_phone'] + df['has_internet'] + df['n_internet_svcs']
        df['no_protect'] = (df['n_protect'] == 0).astype(int)
        df['protect_ratio'] = df['n_protect'] / 4.0
        df['stream_only'] = ((df['n_stream'] > 0) & (df['n_protect'] == 0)).astype(int)
        df['full_protect'] = (df['n_protect'] == 4).astype(int)
        df['cost_per_svc'] = mc / (df['total_svcs'] + 1)

        # Contract & billing
        df['is_mtm'] = (df['Contract'] == 'Month-to-month').astype(int)
        df['is_1yr'] = (df['Contract'] == 'One year').astype(int)
        df['is_2yr'] = (df['Contract'] == 'Two year').astype(int)
        df['paperless'] = (df['PaperlessBilling'] == 'Yes').astype(int)
        df['echeck'] = (df['PaymentMethod'] == 'Electronic check').astype(int)
        df['auto_pay'] = df['PaymentMethod'].isin(
            ['Bank transfer (automatic)', 'Credit card (automatic)']
        ).astype(int)

        # Demographics
        df['senior'] = df['SeniorCitizen'].astype(int)
        df['male'] = (df['gender'] == 'Male').astype(int)
        df['partner'] = (df['Partner'] == 'Yes').astype(int)
        df['dependents'] = (df['Dependents'] == 'Yes').astype(int)
        df['family'] = df['partner'] + df['dependents']
        df['senior_alone'] = (df['senior'] & ~df['partner'].astype(bool)).astype(int)
        df['no_family'] = ((df['partner'] == 0) & (df['dependents'] == 0)).astype(int)

        # Interaction features
        df['fiber_no_protect'] = df['has_fiber'] * df['no_protect']
        df['fiber_mtm'] = df['has_fiber'] * df['is_mtm']
        df['new_fiber'] = df['is_new'] * df['has_fiber']
        df['new_mtm'] = df['is_new'] * df['is_mtm']
        df['echeck_mtm'] = df['echeck'] * df['is_mtm']
        df['hi_charge_mtm'] = df['is_high_charge'] * df['is_mtm']
        df['hi_charge_fiber'] = df['is_high_charge'] * df['has_fiber']
        df['svc_per_charge'] = df['total_svcs'] / (mc + 1e-5)
        df['tenure_x_mc'] = t * mc
        df['tenure_x_mtm'] = t * df['is_mtm']
        df['tenure_x_protect'] = t * df['n_protect']
        df['loyalty'] = t * (1 - df['is_mtm']) * df['auto_pay']
        df['fiber_echeck'] = df['has_fiber'] * df['echeck']
        df['new_echeck'] = df['is_new'] * df['echeck']
        df['fiber_no_support'] = df['has_fiber'] * (1 - df['has_support'])
        df['new_hi_charge'] = df['is_new'] * df['is_high_charge']
        df['loyal_2yr_auto'] = df['is_loyal'] * df['is_2yr'] * df['auto_pay']
        df['senior_mtm'] = df['senior'] * df['is_mtm']
        df['alone_mtm'] = df['no_family'] * df['is_mtm']
        df['fiber_mtm_echeck'] = df['has_fiber'] * df['is_mtm'] * df['echeck']
        df['new_fiber_no_protect'] = df['is_new'] * df['has_fiber'] * df['no_protect']

        # Risk scores
        df['risk_v1'] = (
            df['is_mtm'] * 3 + df['has_fiber'] * 2 + df['echeck'] * 2 +
            df['no_protect'] * 2 + df['is_new'] * 3 + df['senior_alone'] * 1 -
            df['is_2yr'] * 3 - df['auto_pay'] * 2 - df['is_loyal'] * 3
        )
        df['risk_v2'] = (
            df['is_mtm'] * 2 + df['has_fiber'] * 1.5 + df['echeck'] * 1.5 +
            df['no_protect'] * 1.5 + df['paperless'] * 0.5 -
            df['is_2yr'] * 2 - df['auto_pay'] * 1.5 - df['n_protect'] * 0.5
        )
        df['risk_tenure'] = df['risk_v1'] / (t + 1)

    return train, test


train_fe, test_fe = feature_engineering(train_df, test_df)
print(f"Features after FE: train={train_fe.shape}, test={test_fe.shape}")

Features after FE: train=(594194, 116), test=(254655, 114)


## 4. Prepare Features

In [7]:
train_ids = train_fe['id'].values
test_ids = test_fe['id'].values
y = train_fe['Churn'].map({'Yes': 1, 'No': 0}).values

drop_cols = [
    'id', 'Churn', 'Churn_encoded', 'gender', 'Partner', 'Dependents',
    'PhoneService', 'MultipleLines', 'InternetService', 'OnlineSecurity',
    'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV',
    'StreamingMovies', 'Contract', 'PaperlessBilling', 'PaymentMethod'
]

X_train = train_fe.drop(columns=[c for c in drop_cols if c in train_fe.columns])
X_test = test_fe.drop(columns=[c for c in drop_cols if c in test_fe.columns])

common = sorted(set(X_train.columns) & set(X_test.columns))
X_train = X_train[common].astype(np.float32)
X_test = X_test[common].astype(np.float32)

X_train = X_train.fillna(0)
X_test = X_test.fillna(0)

feature_names = list(X_train.columns)
X_tr = X_train.values
X_te = X_test.values

print(f"Final feature count: {len(feature_names)}")
print(f"Train: {X_tr.shape}, Test: {X_te.shape}")
print(f"Positive rate: {y.mean():.4f} ({y.sum()}/{len(y)})")

Final feature count: 98
Train: (594194, 98), Test: (254655, 98)
Positive rate: 0.2252 (133817/594194)


## 5. Optuna Hyperparameter Tuning — LightGBM (GPU Optimized)

In [9]:
def lgb_objective(trial):
    params = {
        'objective': 'binary',
        'metric': 'auc',
        'boosting_type': 'gbdt',
        'verbosity': -1,
        'random_state': SEED,
        'device': LGBM_DEVICE,
        'n_jobs': -1 if LGBM_DEVICE == 'cpu' else 1,
        'learning_rate': trial.suggest_float('learning_rate', 0.005, 0.1, log=True),
        'num_leaves': trial.suggest_int('num_leaves', 31, 511) if HAS_GPU else trial.suggest_int('num_leaves', 31, 127),
        'max_depth': trial.suggest_int('max_depth', -1, 12),
        'min_child_samples': trial.suggest_int('min_child_samples', 20, 200),
        'feature_fraction': trial.suggest_float('feature_fraction', 0.4, 1.0),
        'bagging_fraction': trial.suggest_float('bagging_fraction', 0.4, 1.0),
        'bagging_freq': trial.suggest_int('bagging_freq', 1, 10),
        'reg_alpha': trial.suggest_float('reg_alpha', 1e-4, 20.0, log=True),
        'reg_lambda': trial.suggest_float('reg_lambda', 1e-4, 20.0, log=True),
        'min_split_gain': trial.suggest_float('min_split_gain', 0.0, 0.2),
        'min_child_weight': trial.suggest_float('min_child_weight', 1e-3, 20.0, log=True),
        'path_smooth': trial.suggest_float('path_smooth', 0.0, 20.0),
        'extra_trees': trial.suggest_categorical('extra_trees', [True, False]),
        'n_estimators': 1500,
    }

    skf = StratifiedKFold(n_splits=3, shuffle=True, random_state=SEED)
    aucs = []

    for tri, vai in skf.split(X_tr, y):
        model = lgb.LGBMClassifier(**params)
        model.fit(
            X_tr[tri], y[tri],
            eval_set=[(X_tr[vai], y[vai])],
            callbacks=[lgb.early_stopping(200, verbose=False), lgb.log_evaluation(0)]
        )
        preds = model.predict_proba(X_tr[vai])[:, 1]
        aucs.append(roc_auc_score(y[vai], preds))
        del model; gc.collect()

    mean_auc = np.mean(aucs)
    print(f"  Trial {trial.number}: AUC={mean_auc:.6f}")
    return mean_auc


t0 = time.time()
lgb_study = optuna.create_study(direction='maximize', sampler=TPESampler(seed=SEED, n_startup_trials=10))
lgb_study.optimize(lgb_objective, n_trials=2, gc_after_trial=True)
print(f"\nLightGBM Optuna done in {(time.time()-t0)/60:.1f} min")
print(f"   Best AUC: {lgb_study.best_value:.6f}")
print(f"   Best params: {lgb_study.best_params}")

  Trial 0: AUC=0.914448
  Trial 1: AUC=0.914134

LightGBM Optuna done in 19.4 min
   Best AUC: 0.914448
   Best params: {'learning_rate': 0.015355286838886862, 'num_leaves': 488, 'max_depth': 9, 'min_child_samples': 128, 'feature_fraction': 0.4936111842654619, 'bagging_fraction': 0.49359671220172163, 'bagging_freq': 1, 'reg_alpha': 3.905042189404141, 'reg_lambda': 0.15364863560723963, 'min_split_gain': 0.1416145155592091, 'min_child_weight': 0.0012261243785158802, 'path_smooth': 19.398197043239886, 'extra_trees': True}


## 6. Optuna Hyperparameter Tuning — XGBoost (GPU Optimized)

In [15]:
import xgboost as xgb
import numpy as np
import gc
import time
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score
import optuna
from optuna.samplers import TPESampler

def xgb_objective(trial):
    params = {
        'objective': 'binary:logistic',
        'eval_metric': 'auc',
        'tree_method': 'hist',
        'random_state': SEED,
        'verbosity': 0,
        'nthread': -1,
        'learning_rate': trial.suggest_float('learning_rate', 0.005, 0.1, log=True),
        'max_depth': trial.suggest_int('max_depth', 3, 11),
        'min_child_weight': trial.suggest_float('min_child_weight', 1, 40),
        'subsample': trial.suggest_float('subsample', 0.4, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.4, 1.0),
        'colsample_bylevel': trial.suggest_float('colsample_bylevel', 0.4, 1.0),
        'colsample_bynode': trial.suggest_float('colsample_bynode', 0.4, 1.0),
        'reg_alpha': trial.suggest_float('reg_alpha', 1e-4, 20.0, log=True),
        'reg_lambda': trial.suggest_float('reg_lambda', 1e-4, 20.0, log=True),
        'gamma': trial.suggest_float('gamma', 0.0, 10.0),
        'max_leaves': trial.suggest_int('max_leaves', 0, 511),
        'max_bin': trial.suggest_int('max_bin', 128, 512),
        'grow_policy': trial.suggest_categorical('grow_policy', ['depthwise', 'lossguide']),
        'scale_pos_weight': trial.suggest_float('scale_pos_weight', 0.8, 4.0),
        'n_estimators': 1500,
        'early_stopping_rounds': 50 
    }

    skf = StratifiedKFold(n_splits=3, shuffle=True, random_state=SEED)
    aucs = []

    for tri, vai in skf.split(X_tr, y):
    # Use standard numpy slicing since X_tr is an ndarray
        X_train_fold, X_val_fold = X_tr[tri], X_tr[vai]
        y_train_fold, y_val_fold = y[tri], y[vai]

        model = xgb.XGBClassifier(**params)
    
        model.fit(
        X_train_fold, y_train_fold,
        eval_set=[(X_val_fold, y_val_fold)],
        verbose=False
    )

        preds = model.predict_proba(X_val_fold)[:, 1]
        aucs.append(roc_auc_score(y_val_fold, preds))
        
        del model
        gc.collect()

    return np.mean(aucs)

# --- Execution ---
print("=" * 60)
t0 = time.time()
xgb_study = optuna.create_study(direction='maximize', sampler=TPESampler(seed=SEED, n_startup_trials=10))
xgb_study.optimize(xgb_objective, n_trials=2, gc_after_trial=True) 

print(f"\nXGBoost Optuna done in {(time.time()-t0)/60:.1f} min")
print(f"  Best AUC: {xgb_study.best_value:.6f}")
print(f"  Best params: {xgb_study.best_params}")


XGBoost Optuna done in 13.5 min
  Best AUC: 0.914936
  Best params: {'learning_rate': 0.008661333735273127, 'max_depth': 5, 'min_child_weight': 21.465500833657277, 'subsample': 0.6591670111852694, 'colsample_bytree': 0.5747374841188252, 'colsample_bylevel': 0.7671117368334277, 'colsample_bynode': 0.4836963163912251, 'reg_alpha': 0.0035372645768101335, 'reg_lambda': 0.008751754379612288, 'gamma': 4.56069984217036, 'max_leaves': 402, 'max_bin': 204, 'grow_policy': 'lossguide', 'scale_pos_weight': 0.9486413207039928}


## 7. Optuna Hyperparameter Tuning — CatBoost (GPU Optimized)

In [13]:
def cb_objective(trial):
    params = {
        'loss_function': 'Logloss',
        'eval_metric': 'AUC',
        'random_seed': SEED,
        'task_type': CAT_TASK_TYPE,
        'devices': '0' if HAS_GPU else None,
        'verbose': False,
        'thread_count': -1,
        'iterations': 1500,
        'early_stopping_rounds': 150,
        'learning_rate': trial.suggest_float('learning_rate', 0.005, 0.1, log=True),
        'depth': trial.suggest_int('depth', 4, 11),
        'l2_leaf_reg': trial.suggest_float('l2_leaf_reg', 1e-2, 20.0, log=True),
        'subsample': trial.suggest_float('subsample', 0.4, 1.0),
        'colsample_bylevel': trial.suggest_float('colsample_bylevel', 0.4, 1.0) if not HAS_GPU else 1.0,
        'min_data_in_leaf': trial.suggest_int('min_data_in_leaf', 5, 200),
        'random_strength': trial.suggest_float('random_strength', 0.0, 20.0),
        'bagging_temperature': trial.suggest_float('bagging_temperature', 0.0, 10.0),
        'border_count': trial.suggest_int('border_count', 32, 255),
        'max_ctr_complexity': trial.suggest_int('max_ctr_complexity', 1, 3),
        'leaf_estimation_iterations': trial.suggest_int('leaf_estimation_iterations', 1, 15),
        'bootstrap_type': trial.suggest_categorical('bootstrap_type', ['Bayesian', 'Bernoulli']) if HAS_GPU else 'MVS',
    }
    
    # Patch bootstrap mapping
    if params['bootstrap_type'] == 'Bayesian':
        params.pop('subsample', None)
    elif params['bootstrap_type'] == 'Bernoulli':
        params.pop('bagging_temperature', None)

    skf = StratifiedKFold(n_splits=3, shuffle=True, random_state=SEED)
    aucs = []

    for tri, vai in skf.split(X_tr, y):
        model = cb.CatBoostClassifier(**params)
        model.fit(X_tr[tri], y[tri], eval_set=(X_tr[vai], y[vai]), verbose=0)
        preds = model.predict_proba(X_tr[vai])[:, 1]
        aucs.append(roc_auc_score(y[vai], preds))
        del model; gc.collect()

    mean_auc = np.mean(aucs)
    print(f"  Trial {trial.number}: AUC={mean_auc:.6f}")
    return mean_auc


print("=" * 60)
t0 = time.time()
cb_study = optuna.create_study(direction='maximize', sampler=TPESampler(seed=SEED, n_startup_trials=10))
cb_study.optimize(cb_objective, n_trials=2, gc_after_trial=True)
print(f"\nCatBoost Optuna done in {(time.time()-t0)/60:.1f} min")
print(f"   Best AUC: {cb_study.best_value:.6f}")
print(f"   Best params: {cb_study.best_params}")

Default metric period is 5 because AUC is/are not implemented for GPU
Default metric period is 5 because AUC is/are not implemented for GPU
Default metric period is 5 because AUC is/are not implemented for GPU


  Trial 0: AUC=0.914773


Default metric period is 5 because AUC is/are not implemented for GPU
Default metric period is 5 because AUC is/are not implemented for GPU
Default metric period is 5 because AUC is/are not implemented for GPU


  Trial 1: AUC=0.915436

CatBoost Optuna done in 3.5 min
   Best AUC: 0.915436
   Best params: {'learning_rate': 0.060534484680010825, 'depth': 5, 'l2_leaf_reg': 0.039829941698924086, 'subsample': 0.5100427059120604, 'min_data_in_leaf': 64, 'random_strength': 10.495128632644757, 'bagging_temperature': 4.319450186421157, 'border_count': 97, 'max_ctr_complexity': 2, 'leaf_estimation_iterations': 3, 'bootstrap_type': 'Bernoulli'}


## 8. Full 3-Fold CV Training — LightGBM

In [16]:
print("LightGBM — Full 5-Fold CV with tuned hyperparameters")
print("=" * 60)

lgb_best = lgb_study.best_params.copy()
lgb_best.update({
    'objective': 'binary',
    'metric': 'auc',
    'boosting_type': 'gbdt',
    'verbosity': -1,
    'random_state': SEED,
    'device': LGBM_DEVICE,
    'n_jobs': -1 if LGBM_DEVICE == 'cpu' else 1,
    'n_estimators': 1500,
})

lgb_oof = np.zeros(len(y))
lgb_test_preds = np.zeros(len(X_te))

skf = StratifiedKFold(n_splits=3, shuffle=True, random_state=SEED)
for fold, (tri, vai) in enumerate(skf.split(X_tr, y)):
    print(f"  Fold {fold+1}/{N_FOLDS}", end="")
    model = lgb.LGBMClassifier(**lgb_best)
    model.fit(
        X_tr[tri], y[tri],
        eval_set=[(X_tr[vai], y[vai])],
        callbacks=[lgb.early_stopping(300, verbose=False), lgb.log_evaluation(0)]
    )
    lgb_oof[vai] = model.predict_proba(X_tr[vai])[:, 1]
    lgb_test_preds += model.predict_proba(X_te)[:, 1] / N_FOLDS
    auc = roc_auc_score(y[vai], lgb_oof[vai])
    print(f" — AUC: {auc:.6f} (best_iter: {model.best_iteration_})")
    del model; gc.collect()

lgb_oof_auc = roc_auc_score(y, lgb_oof)
print(f"\nLightGBM OOF AUC: {lgb_oof_auc:.6f}")

LightGBM — Full 5-Fold CV with tuned hyperparameters
  Fold 1/5 — AUC: 0.914293 (best_iter: 1500)
  Fold 2/5 — AUC: 0.915296 (best_iter: 1500)
  Fold 3/5 — AUC: 0.913848 (best_iter: 1500)

LightGBM OOF AUC: 0.914474


## 9. Full 5-Fold CV Training — XGBoost

In [22]:
print("XGBoost — Full 3-Fold CV (Fixed Iterations)")
print("=" * 60)

# 1. Update params to ensure consistency
xgb_params = xgb_study.best_params.copy()
xgb_params.update({
    'objective': 'binary:logistic',
    'eval_metric': 'auc',
    'tree_method': 'hist',
    'random_state': SEED,
    'verbosity': 0,
    'nthread': -1,
    'n_estimators': 1200, # Fixed rounds
})

xgb_oof = np.zeros(len(y))
xgb_test_preds = np.zeros(len(X_te))

skf = StratifiedKFold(n_splits=3, shuffle=True, random_state=SEED)

for fold, (tri, vai) in enumerate(skf.split(X_tr, y)):
    print(f"  Fold {fold+1}/3", end="")
    
    # 2. Initialize and fit without early_stopping_rounds or eval_set
    model = xgb.XGBClassifier(**xgb_params)
    model.fit(X_tr[tri], y[tri], verbose=False)
    
    # 3. Predict OOF and Test
    xgb_oof[vai] = model.predict_proba(X_tr[vai])[:, 1]
    xgb_test_preds += model.predict_proba(X_te)[:, 1] / 3
    
    # 4. Metrics - Removed best_iteration as it is now undefined
    auc = roc_auc_score(y[vai], xgb_oof[vai])
    print(f" — AUC: {auc:.6f}")
    
    del model
    gc.collect()

xgb_oof_auc = roc_auc_score(y, xgb_oof)
print(f"\nXGBoost OOF AUC: {xgb_oof_auc:.6f}")

XGBoost — Full 3-Fold CV (Fixed Iterations)
  Fold 1/3 — AUC: 0.914224
  Fold 2/3 — AUC: 0.915781
  Fold 3/3 — AUC: 0.914147

XGBoost OOF AUC: 0.914666


## 10. Full 5-Fold CV Training — CatBoost

In [24]:
print("CatBoost — Full 3-Fold CV (Fixed Parameters)")
print("=" * 60)

cb_params = cb_study.best_params.copy()

# Fix the Bagging Temperature error
# If bootstrap_type is not 'Bayesian', bagging_temperature must be removed
if cb_params.get('bootstrap_type') != 'Bayesian':
    cb_params.pop('bagging_temperature', None)

cb_params.update({
    'loss_function': 'Logloss',
    'eval_metric': 'AUC',
    'random_seed': SEED,
    'task_type': CAT_TASK_TYPE,
    'devices': '0' if (HAS_GPU and CAT_TASK_TYPE == 'GPU') else None,
    'verbose': False,
    'thread_count': -1,
    'iterations': 1500,
})

# REMOVED: early_stopping_rounds to follow your previous requirement
cb_params.pop('early_stopping_rounds', None)

cb_oof = np.zeros(len(y))
cb_test_preds = np.zeros(len(X_te))

skf = StratifiedKFold(n_splits=3, shuffle=True, random_state=SEED)

for fold, (tri, vai) in enumerate(skf.split(X_tr, y)):
    print(f"  Fold {fold+1}/3", end="")
    
    model = cb.CatBoostClassifier(**cb_params)
    
    # REMOVED: eval_set and early stopping logic
    model.fit(X_tr[tri], y[tri], verbose=0)
    
    cb_oof[vai] = model.predict_proba(X_tr[vai])[:, 1]
    cb_test_preds += model.predict_proba(X_te)[:, 1] / 3
    
    auc = roc_auc_score(y[vai], cb_oof[vai])
    
    # REMOVED: model.get_best_iteration() as it is not applicable without eval_set
    print(f" — AUC: {auc:.6f}")
    
    del model
    gc.collect()

cb_oof_auc = roc_auc_score(y, cb_oof)
print(f"\nCatBoost OOF AUC: {cb_oof_auc:.6f}")

CatBoost — Full 3-Fold CV (Fixed Parameters)
  Fold 1/3

Default metric period is 5 because AUC is/are not implemented for GPU


 — AUC: 0.915266
  Fold 2/3

Default metric period is 5 because AUC is/are not implemented for GPU


 — AUC: 0.916245
  Fold 3/3

Default metric period is 5 because AUC is/are not implemented for GPU


 — AUC: 0.914666

CatBoost OOF AUC: 0.915385


## 11. Optimal Blending (AUC Optimized)

In [25]:
print("Finding Optimal Blend Weights")
print("=" * 60)

models_info = [
    ('LightGBM', lgb_oof, lgb_test_preds),
    ('XGBoost',  xgb_oof, xgb_test_preds),
    ('CatBoost', cb_oof,  cb_test_preds),
]

oof_stack = np.column_stack([m[1] for m in models_info])
test_stack = np.column_stack([m[2] for m in models_info])


def neg_auc(w):
    """Negative AUC for minimization."""
    w_norm = np.abs(w) / np.abs(w).sum()
    blend = oof_stack @ w_norm
    return -roc_auc_score(y, blend)


# Scipy optimizer to find best weights
w0 = np.ones(len(models_info)) / len(models_info)
result = minimize(neg_auc, w0, method='Nelder-Mead',
                  options={'maxiter': 10000, 'xatol': 1e-8, 'fatol': 1e-8})
best_weights = np.abs(result.x) / np.abs(result.x).sum()

# Compute blended predictions
blend_oof = oof_stack @ best_weights
final_test_preds = test_stack @ best_weights

print("\nIndividual OOF AUCs:")
for name, oof, _ in models_info:
    print(f"   {name:12s}: {roc_auc_score(y, oof):.6f}")

print(f"\nOptimal Blend Weights:")
for (name, _, _), w in zip(models_info, best_weights):
    print(f"   {name:12s}: {w:.4f}")

blend_auc = roc_auc_score(y, blend_oof)
print(f"\nBlended OOF AUC: {blend_auc:.6f}")

# Quick sanity check — also try equal weights
equal_blend = oof_stack.mean(axis=1)
equal_auc = roc_auc_score(y, equal_blend)
print(f"   Equal-weight AUC: {equal_auc:.6f}")

if equal_auc > blend_auc:
    print("Equal weights beat optimized — using equal weights (more robust)")
    final_test_preds = test_stack.mean(axis=1)
    blend_auc = equal_auc

Finding Optimal Blend Weights

Individual OOF AUCs:
   LightGBM    : 0.914474
   XGBoost     : 0.914666
   CatBoost    : 0.915385

Optimal Blend Weights:
   LightGBM    : 0.0000
   XGBoost     : 0.1990
   CatBoost    : 0.8010

Blended OOF AUC: 0.915431
   Equal-weight AUC: 0.915175


## 12. Create Submission

In [26]:
submission = pd.DataFrame({'id': test_ids, 'Churn': final_test_preds})

out_path = '/kaggle/working/submission.csv' if os.path.exists('/kaggle/working') else 'submission.csv'
submission.to_csv(out_path, index=False)

print(f"Submission saved to {out_path}")
print(f"   Shape: {submission.shape}")
print(f"\n{submission.head(10)}")
print(f"\nPrediction statistics:\n{submission['Churn'].describe()}")

Submission saved to /kaggle/working/submission.csv
   Shape: (254655, 2)

       id     Churn
0  594194  0.066424
1  594195  0.001270
2  594196  0.092768
3  594197  0.003016
4  594198  0.483040
5  594199  0.163486
6  594200  0.878301
7  594201  0.002511
8  594202  0.022301
9  594203  0.326688

Prediction statistics:
count    254655.000000
mean          0.217365
std           0.273257
min           0.000761
25%           0.006564
50%           0.064266
75%           0.390413
max           0.985417
Name: Churn, dtype: float64
